# 导入相关的包

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass
torch.manual_seed(1)
import math

In [3]:
# 定义相关的参数

@dataclass
class GPTConfig:
    vocab_size: int = 50257  # 词表大小
    block_size: int = 512   # 上下文窗口大小。文本的最大长度
    batch_size: int = 12    # 批量大小
    n_layer: int = 12       # Transformer 层数
    n_head: int = 12        # 注意力头数
    n_embd: int = 768        # 嵌入维度
    hidden_dim: int= n_embd
    # 为了可以tie_embeddings_weight

    head_size: int = n_embd//n_head      # 注意力头大小

    dropout: float = 0.1      # dropout 概率


# 3. 定义GPT的结构

In [ ]:
class SingleHeadAttention(nn.Module):
    # 定义一个单头注意力机制，继承自nn.Module
    def __init__(self, config: GPTConfig):
        # 构造函数，接收一个配置对象
        super().__init__()
        # 调用父类的初始化方法，确保模块正常注册
        self.key = nn.Linear(config.hidden_dim, config.head_size, bias=False)
        # 定义一个线性层，将输入的隐藏状态映射为 key 向量。
        self.query = nn.Linear(config.hidden_dim, config.head_size, bias=False)
        # 定义一个线性层，将输入的隐藏状态映射为 query 向量。
        self.value = nn.Linear(config.hidden_dim, config.head_size, bias=False)
        # 定义一个线性层，将输入的隐藏状态映射为 value 向量。



        self.register_buffer("attention_mask", torch.tril(torch.ones(config.block_size, config.block_size)))
        # 尝试attention masking的新写法，attention mask通过register_buffer注册
        # 注册一个下三角矩阵（掩码），用于防止模型看到未来的信息（自回归）。tril代表下三角矩阵。对角线及以下为1，其他为0。
        # register_buffer继承自pytorch的nn.Module，可以将一个张量注册为模块的缓冲区，
        # block_size 是文本的最大长度，生成一个 block_size x block_size 的下三角矩阵，1 表示允许关注的位置，0 表示不允许关注的位置。

        self.dropout = nn.Dropout(config.dropout)
        # 定义一个 dropout 层，用于防止过拟合。

    def forward(self, x):
        # 前向传播函数，输入为张量 x。
        batch_size, seq_len, hidden_dim = x.shape
        k = self.key(x)      # (B,T,head_size)
        # 获取输入的 batch 大小、序列长度、特征维度。
        q = self.query(x)    # (B,T,head_size)
        # 通过线性层得到 query 向量，形状为 (B, T, head_size)。
        v = self.value(x)    # (B,T,head_size)
        # 通过线性层得到 value 向量，形状为 (B, T, head_size)。


        # 计算注意力得分
        weight = q @ k.transpose(-2, -1)
        # 在 Python 和 PyTorch 中，@ 是矩阵乘法运算符（matmul）。对每个 batch，计算每个时间步的 query 与所有时间步的 key 的点积，得到 (B, T, T) 的注意力分数矩阵。
        # 计算 query 和 key 的点积，除以根号下 head_size 进行缩放，得到注意力分数。
        # transpose(-2, -1) 将 key 的最后两个维度进行转置，以便进行矩阵乘法。形状为 (B, T, head_size)。
        # 乘以缩放因子 (1.0 / sqrt(head_size))，防止点积值过大导致梯度消失或爆炸。


        weight = weight.masked_fill(self.attention_mask[:seq_len, :seq_len] == 0, float('-inf'))
        # "attention_mask" 是一个下三角矩阵，用于掩码未来位置的注意力分数。引用注册的缓冲区。


        weight = F.softmax(weight, dim=-1)/ math.sqrt(k.size(-1))
        # 通过 F 使用 torch.nn.functional 里的函数，例如 F.softmax、F.relu 等
        # 用下三角掩码将未来位置的分数设为负无穷，防止模型关注未来。
        # 对最后一个维度进行 softmax，使注意力分数归一化。


        weight = self.dropout(weight)
        # 对注意力分数应用 dropout。


        y = weight @ v      # (B,T,head_size)


        return y

输入 x (batch_size, seq_len, hidden_dim)
      │
      ▼
+-----------------------------+
| 线性层: self.key            |
| 线性层: self.query          |
| 线性层: self.value          |
+-----------------------------+
      │         │         │
      ▼         ▼         ▼
   k (B,T,H)  q (B,T,H)  v (B,T,H)
      │         │
      └───┬─────┘
          │
          ▼
点积注意力计算: weight = q @ k^T
          │
          ▼
掩码处理: weight = weight.masked_fill(mask==0, -inf)
          │
          ▼
Softmax归一化: weight = softmax(weight, dim=-1) / sqrt(H)
          │
          ▼
Dropout: weight = dropout(weight)
          │
          ▼
加权求和: y = weight @ v
          │
          ▼
输出: y (B, T, H)

In [ ]:
111
